In [ ]:
# =====================================================
# Skin Disease Detection using CNN (Custom Model)
# =====================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

In [ ]:
# -------------------------
# 1️⃣ Paths and Parameters
# -------------------------
# Use relative paths or environment variables instead of hardcoded paths
import os

# Update these paths to point to your kaggle dataset
train_dir = os.getenv('TRAIN_DIR', './data/train')      # path to your training data
val_dir = os.getenv('VAL_DIR', './data/val')            # path to your validation data

IMG_SIZE = (128, 128)  # IMPORTANT: Keep consistent across all scripts
BATCH_SIZE = 32
EPOCHS = 1

In [ ]:

# -------------------------
# 2️⃣ Data Augmentation
# -------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


In [ ]:

# Number of classes
num_classes = len(train_generator.class_indices)
print("Detected Classes:", train_generator.class_indices)


In [ ]:

# -------------------------
# 3️⃣ CNN Architecture
# -------------------------
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.3),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.4),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])


In [ ]:

# -------------------------
# 4️⃣ Compile the Model
# -------------------------
optimizer = optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:

# -------------------------
# 5️⃣ Callbacks for Stability
# -------------------------
cb_early = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
cb_reduce = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
cb_checkpoint = callbacks.ModelCheckpoint("best_skin_model.keras", monitor='val_loss', save_best_only=True)


In [ ]:

# -------------------------
# 6️⃣ Train the Model
# -------------------------
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[cb_early, cb_reduce, cb_checkpoint]
)


In [ ]:

# -------------------------
# 7️⃣ Evaluate Model
# -------------------------
val_loss, val_acc = model.evaluate(val_generator)
print(f"✅ Validation Accuracy: {val_acc*100:.2f}%")


In [ ]:

# -------------------------
# 8️⃣ Plot Accuracy & Loss
# -------------------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Directory where test/validation images are stored
test_dir = os.getenv('VAL_DIR', './data/val')  # Use same path as validation

# Preprocessing (rescale pixel values)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create validation generator - IMPORTANT: Use IMG_SIZE (128, 128)
validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,  # Match your CNN input size (128, 128)
    batch_size=32,
    class_mode='categorical',  # use 'categorical' if one-hot encoded
    shuffle=False
)

In [ ]:
# Save the model in .h5 format for consistency with predict.py
model.save('skin_disease_model.h5')

# Also save class indices for use in prediction
import json
with open('labels.json', 'w') as f:
    json.dump(list(train_generator.class_indices.keys()), f)

print("✅ Model saved as 'skin_disease_model.h5'")
print("✅ Class labels saved as 'labels.json'")